# 11 — Static Validation (Synthetic Expert Data)

**Question:** Can GS-CV and static SBI correctly identify BE vs SC
and recover parameters from expert-phase data with known ground truth?

**Protocol:**
1. Generate synthetic cohort: N BE + N SC animals, static expert params,
   uniform distribution, 15 sessions × 350 trials each
2. Run GS-CV on each (model selection + parameter recovery)
3. Run static SBI on each (same metrics)
4. Compare: accuracy, recovery correlation, confusion matrices

All ground truth is known — this is a pure methods validation.

In [ ]:
%matplotlib inline
from shared_setup import *

import time
import pickle
from analysis.validation import (
    make_synthetic_cohort,
    run_gs_model_id,
)
from scripts.config import (
    SYNTH_N_PER_MODEL, SYNTH_N_SESSIONS, SYNTH_TRIALS_PER_SESSION,
    GS_BURN_IN, BASE_SEED, GS_N_SEEDS, SYNTH_GS_N_SEEDS,
    SYNTH_COHORTS_DIR,
)

SBI_OK = False
try:
    import torch
    from inference.amortised import AmortisedSBI
    SBI_OK = True
    print(f'SBI available (torch {torch.__version__})')
except ImportError as e:
    print(f'SBI not available: {e}')

apply_style()

## 1. Configuration

In [ ]:
# Toggle for fast iteration vs full run
QUICK = True

if QUICK:
    N_PER_MODEL = 1
    N_GS_SEEDS = 4
    N_SBI_SIMS = 2_000
    N_SBI_REPEATS = 4          
    N_SBI_POSTERIOR = 30
    N_SBI_STOCHASTIC = 3       
    BURN_IN = 500
else:
    N_PER_MODEL = SYNTH_N_PER_MODEL
    N_GS_SEEDS = SYNTH_GS_N_SEEDS
    N_SBI_SIMS = 50_000
    N_SBI_REPEATS = 64         
    N_SBI_POSTERIOR = 50
    N_SBI_STOCHASTIC = 10
    BURN_IN = GS_BURN_IN

N_SESSIONS = SYNTH_N_SESSIONS
TRIALS_PER_SESSION = SYNTH_TRIALS_PER_SESSION
SEED = BASE_SEED
FIT_TARGET = 'update_matrix'

print(f'Mode: {"QUICK" if QUICK else "FULL"}')
print(f'Animals: {N_PER_MODEL} BE + {N_PER_MODEL} SC')
print(f'Sessions: {N_SESSIONS} × {TRIALS_PER_SESSION} trials')

## 2. Generate synthetic cohort

In [ ]:
animals = make_synthetic_cohort(
    n_per_model=N_PER_MODEL,
    n_sessions=N_SESSIONS,
    trials_per_session=TRIALS_PER_SESSION,
    burn_in=BURN_IN,
    seed=SEED,
)

print(f'{len(animals)} synthetic animals generated')
for a in animals:
    aid = a['animal_id']
    model = a['true_model']
    params = a['true_params']
    n_sess = len(a['sessions'])
    print(f'  {aid}: {model}, {n_sess} sessions, {params}')

## 3. Grid-search model identification

Uses `run_gs_model_id` from `analysis/validation.py`, which
internally calls `grid_search_cv` (the same function used by
`scripts/grid_search.py`). Runs both BE and SC fits for each
animal and determines the winner via Wilcoxon on CV errors.

In [ ]:
print('=== Grid-Search Model Identification ===')
t0 = time.time()

gs_df = run_gs_model_id(
    animals,
    sessions_key='sessions',
    n_seeds=N_GS_SEEDS,
    burn_in=BURN_IN,
    fit_target=FIT_TARGET,
)

dt = time.time() - t0
gs_acc = gs_df['gs_correct'].mean()
print(f'\nAccuracy: {gs_df["gs_correct"].sum()}/{len(gs_df)} '
      f'({gs_acc:.0%}) in {dt:.1f}s')

In [ ]:
# Per-animal results
display_cols = ['animal_id', 'true_model', 'gs_winner', 'gs_correct',
                'gs_be_mean', 'gs_sc_mean']
available = [c for c in display_cols if c in gs_df.columns]
gs_df[available]

### 3b. GS parameter recovery

For correctly identified animals, compare the best-fit parameters
(from the best grid point) against the true generating parameters.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

for model_type in ['BE', 'SC']:
    mask = (gs_df['true_model'] == model_type) & gs_df['gs_correct']
    subset = gs_df[mask]
    if subset.empty:
        continue

    if model_type == 'BE':
        param_names = ['sigma_percep', 'A_repulsion', 'eta_learning', 'eta_relax']
    else:
        param_names = ['sigma_percep', 'A_repulsion', 'gamma', 'sigma_update']

    for ax, pn in zip(axes, param_names):
        true_vals = [a['true_params'].__dict__[pn]
                     for _, a in zip(range(len(subset)),
                     [a for a in animals if a['true_model'] == model_type
                      and a['animal_id'] in subset['animal_id'].values])]
        recovered = subset['gs_recovered_params'].apply(
            lambda d: d.get(pn, np.nan)).values

        n_plot = min(len(true_vals), len(recovered))
        colour = COLOURS.get(model_type, 'grey')
        ax.scatter(true_vals[:n_plot], recovered[:n_plot],
                   c=colour, s=40, alpha=0.7, label=model_type)
        lims = ax.get_xlim()
        ax.plot(lims, lims, 'k--', lw=0.5, alpha=0.5)
        ax.set_xlabel(f'True {pn}')
        ax.set_ylabel(f'Recovered {pn}')
        ax.legend(fontsize=8)

fig.suptitle('GS Parameter Recovery (correct animals only)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Static SBI model identification

Uses `AmortisedSBI` from `inference/amortised.py`:
- Train once per model type on curriculum-matched generic data
- Condition on each animal → held-out CV errors
- Compare BE vs SC errors via Wilcoxon

In [ ]:
if not SBI_OK:
    print('Skipping SBI (torch not available)')
else:
    from scripts.compare_models import compare_results

    curriculum = [('uniform', N_SESSIONS)]

    # Train BE and SC amortised networks
    print('=== Training AmortisedSBI ===')
    trainers = {}
    for mt in ['be', 'sc']:
        t0 = time.time()
        trainer = AmortisedSBI(
            model_type=mt,
            curriculum=curriculum,
            trials_per_session=TRIALS_PER_SESSION,
            burn_in=BURN_IN,
        )
        trainer.train(n_simulations=N_SBI_SIMS, seed=SEED)
        trainers[mt] = trainer
        print(f'  {mt.upper()}: {time.time() - t0:.1f}s')

In [ ]:
if SBI_OK:
    print('=== Conditioning on synthetic animals ===')

    sbi_rows = []
    for a in animals:
        aid = a['animal_id']
        sessions = a['sessions']

        # Fit with both models
        results = {}
        for mt in ['be', 'sc']:
            results[mt] = trainers[mt].fit(
				sessions=sessions,
				animal_id=aid,
				fit_target=FIT_TARGET,
				n_repeats=N_SBI_REPEATS,
				n_posterior_samples=N_SBI_POSTERIOR,
				n_stochastic_reps=N_SBI_STOCHASTIC,
			)

        # Compare
        comp = compare_results(results['be'], results['sc'])

        sbi_rows.append({
            'animal_id': aid,
            'true_model': a['true_model'],
            'sbi_winner': comp['winner'],
            'sbi_correct': comp['winner'] == a['true_model'],
            'sbi_p_value': comp['p_value'],
            'sbi_be_mean': comp['be_mean_error'],
            'sbi_sc_mean': comp['sc_mean_error'],
            'sbi_be_params': results['be']['best_params'],
            'sbi_sc_params': results['sc']['best_params'],
        })
        status = '✓' if comp['winner'] == a['true_model'] else '✗'
        print(f'  {aid}: true={a["true_model"]}, '
              f'pred={comp["winner"]} {status} '
              f'(p={comp["p_value"]:.3f})')

    sbi_df = pd.DataFrame(sbi_rows)
    sbi_acc = sbi_df['sbi_correct'].mean()
    print(f'\nAccuracy: {sbi_df["sbi_correct"].sum()}/{len(sbi_df)} '
          f'({sbi_acc:.0%})')

### 4b. SBI parameter recovery

For correctly identified animals, compare posterior median parameters
against truth. SBI should outperform GS here because it estimates
continuous parameters rather than selecting from a discrete grid.

In [ ]:
if SBI_OK:
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

    for model_type in ['BE', 'SC']:
        mask = (sbi_df['true_model'] == model_type) & sbi_df['sbi_correct']
        subset = sbi_df[mask]
        if subset.empty:
            continue

        mt_lower = model_type.lower()
        if model_type == 'BE':
            param_names = ['sigma_percep', 'A_repulsion',
                           'eta_learning', 'eta_relax']
        else:
            param_names = ['sigma_percep', 'A_repulsion',
                           'gamma', 'sigma_update']

        matched_animals = [a for a in animals
                           if a['true_model'] == model_type
                           and a['animal_id'] in subset['animal_id'].values]

        for ax, pn in zip(axes, param_names):
            true_vals = [a['true_params'].__dict__[pn]
                         for a in matched_animals]
            recovered = subset[f'sbi_{mt_lower}_params'].apply(
                lambda d: d.get(pn, np.nan)).values

            n_plot = min(len(true_vals), len(recovered))
            colour = COLOURS.get(model_type, 'grey')
            ax.scatter(true_vals[:n_plot], recovered[:n_plot],
                       c=colour, s=40, alpha=0.7, label=model_type)
            lims = ax.get_xlim()
            ax.plot(lims, lims, 'k--', lw=0.5, alpha=0.5)
            ax.set_xlabel(f'True {pn}')
            ax.set_ylabel(f'Recovered {pn}')
            ax.legend(fontsize=8)

    fig.suptitle('SBI Parameter Recovery (correct animals only)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 5. Head-to-head comparison

Compare GS and SBI on the same synthetic cohort.

In [ ]:
if SBI_OK:
    merged = gs_df.merge(sbi_df, on=['animal_id', 'true_model'])

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # --- Panel 1: Accuracy bar chart ---
    ax = axes[0]
    methods = ['GS', 'SBI']
    accs = [gs_acc, sbi_acc]
    bars = ax.bar(methods, accs, color=[PALETTE[0], PALETTE[1]], width=0.5)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Model ID Accuracy')
    ax.axhline(0.5, ls=':', color='grey', lw=0.8)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width() / 2, acc + 0.03,
                f'{acc:.0%}', ha='center', fontsize=11)
    ax.set_title('Model Selection Accuracy')

    # --- Panel 2: Confusion matrix (GS) ---
    ax = axes[1]
    from sklearn.metrics import confusion_matrix
    cm_gs = confusion_matrix(
        merged['true_model'], merged['gs_winner'], labels=['BE', 'SC'])
    im = ax.imshow(cm_gs, cmap='Blues', vmin=0)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['BE', 'SC'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['BE', 'SC'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm_gs[i, j]), ha='center', va='center',
                    fontsize=14, color='white' if cm_gs[i, j] > cm_gs.max() / 2 else 'black')
    ax.set_title('GS Confusion Matrix')

    # --- Panel 3: Confusion matrix (SBI) ---
    ax = axes[2]
    cm_sbi = confusion_matrix(
        merged['true_model'], merged['sbi_winner'], labels=['BE', 'SC'])
    im = ax.imshow(cm_sbi, cmap='Oranges', vmin=0)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['BE', 'SC'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['BE', 'SC'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm_sbi[i, j]), ha='center', va='center',
                    fontsize=14, color='white' if cm_sbi[i, j] > cm_sbi.max() / 2 else 'black')
    ax.set_title('SBI Confusion Matrix')

    plt.tight_layout()
    plt.show()
else:
    print('SBI not available — showing GS results only')
    print(f'GS accuracy: {gs_acc:.0%}')

## 6. Summary

**Expected results** (from previous validation runs, N=20 per model):
- GS accuracy: 76–92% (depends on parameter regime)
- SBI accuracy: 92–100%
- SBI parameter recovery substantially better for continuous params
  (especially η_learning: r ≈ 0.84 vs GS r ≈ 0.19)
- Both methods struggle with the same animals: those with parameters
  near the BE/SC decision boundary where UMs look similar

**Key takeaways:**
- SBI complements GS — higher accuracy and better parameter recovery
- GS remains a valuable sanity check (simpler, well-understood)
- Using both in a consensus framework is more robust than either alone

**Next:** NB 12 validates the dynamic methods. NB 13 tests the full
trajectory scenario (learning → expert → shift).